# 0.导入包 和 构造函数

In [50]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
pd.options.display.float_format = '{:.6f}'.format  # 保留6位小数

import warnings
warnings.filterwarnings("ignore")
# 设置中文字体为 SimHei（黑体）
plt.rcParams['font.sans-serif'] = ['SimHei']  # 使用黑体 SimHei
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题

# 1.读入数据（以23年数据为基础）

## 1.1 城市不同距离等级路网（cities_flows）载货量（Wton）：

In [52]:
#--------------城市5距离区间货运量 =  未来城市总公路货运量 * 未来城市的5个距离区间货运量占比
cities_flows_23 = pd.read_csv("../data/dataset2.MultiDistance_2023地级市不同里程等级货运量.csv")
cities_flows_23


,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en
0,北京市,2023,19399.000009,intra-city,0.407323,0.269966,5237.075531,Beijing
1,北京市,2023,19399.000009,10-50,0.034243,0.278118,5395.217434,Beijing
2,北京市,2023,19399.000009,50-100,0.000000,0.000000,0.000000,Beijing
3,北京市,2023,19399.000009,100-200,0.412020,0.346555,6722.818816,Beijing
4,北京市,2023,19399.000009,200-400,0.054448,0.045220,877.219386,Beijing
...,...,...,...,...,...,...,...,...
1705,哈密市,2023,34773.493677,10-50,0.000000,0.000000,0.000000,Hami
1706,哈密市,2023,34773.493677,50-100,0.000000,0.000000,0.000000,Hami
1707,哈密市,2023,34773.493677,100-200,0.000000,0.000000,0.000000,Hami
1708,哈密市,2023,34773.493677,200-400,0.000000,0.000000,0.000000,Hami


## 1.2 城市的新能源货车渗透率：城市的新能源货车渗透率在（基准情景） * （2023估计13-22）

In [53]:
new_energy_rate_2023 = pd.read_excel("../data/城市新能源渗透率3.0.xlsx",sheet_name = "2023")

new_energy_rate_2023 = new_energy_rate_2023.iloc[::,:2].rename(columns={'城市': 'city'})

new_energy_rate_2023

,city,基准新能源占比
0,北京市,0.165930
1,天津市,0.113599
2,石家庄市,0.045814
3,唐山市,0.042218
4,秦皇岛市,0.024561
...,...,...
361,铁门关市,0.010147
362,双河市,0.010068
363,可克达拉市,0.009831
364,昆玉市,0.009576


## 为不同车型设计出不同渗透率表格，所有数据乘以系数 = 把不同车型的新能源渗透率比例关系找出来就可以得到这个系数

In [54]:
# 以上是属于5吨的那一档，还得为不同车型设计出不同渗透率表格，所有数据乘以系数 = 把不同车型的新能源渗透率比例关系找出来就可以得到这个系数
#https://www.chinatruck.org/news/202311/68_115111.html
#2023各类型卡车新能源渗透率占比
dist_factor = {
    "10-50": 6.79,#5吨微卡
    "50-100": 2.55,
    "100-200": 12.38,
    "200-400": 4.06,
    "400+": 5.58,
    "intra-city": 6.79
}

In [56]:
import pandas as pd
import numpy as np

def build_ev_rate_by_distance(df,
                              dist_factor=None,
                              cap=0.99,
                              normalize="by_0_50"):
    """
    将“城市级（5吨档）新能源渗透率表”扩展为“城市×距离段×情景”的渗透率表。

    参数
    ----
    df : DataFrame
        必含列：['city','基准新能源占比']
    dist_factor : dict or None
        距离段→相对系数（原始权重）。默认：
        {"0-50":6.79,"50-100":2.55,"100-200":12.38,"200-400":4.06,"400+":5.58}
    cap : float
        上限截断，避免比例超过 1（默认 0.99）
    normalize : {"by_0_50","mean",None}
        归一方式：
        - "by_0_50": 以 0-50 为 1，其它相对它缩放（保持 0-50 不变）
        - "mean": 使各系数的算术平均=1（不改变总体均值，只改相对强弱）
        - None: 不归一，直接使用原系数（谨慎，可能直接“爆表”）

    返回
    ----
    DataFrame: 列包含
      ['city','distance_group','mult',
       '基准新能源占比','基准燃油占比',
       '低新能源占比','低新能源燃油占比',
       '高新能源占比','高新能源燃油占比']
    """
    # ---- 必要列校验 ----
    required = ['city','基准新能源占比']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"输入 df 缺少必要列：{missing}")

    # ---- 默认距离段系数 ----
    if dist_factor is None:
        dist_factor = {"10-50": 6.79, "50-100": 2.55, "100-200": 12.38, "200-400": 4.06, "400+": 5.58,"intra-city": 6.79}

    factors = pd.Series(dist_factor, name="mult")

    # ---- 归一化 ----
    if normalize == "by_10_50":
        if "10-50" not in factors.index:
            raise ValueError("normalize='by_10_50' 需要 dist_factor 含 '10-50' 键")
        factors = factors / factors.loc["10-50"]
    elif normalize == "mean":
        factors = factors / factors.mean()
    elif normalize is None:
        pass
    else:
        raise ValueError("normalize 仅支持 {'by_10_50','mean',None}")

    # ---- 笛卡尔积：城市 × 距离段 ----
    left = df.copy()
    left["key"] = 1
    right = factors.reset_index().rename(columns={"index":"distance_group"})
    right["key"] = 1
    out = left.merge(right, on="key").drop(columns="key")

    # ---- 按距离段倍率调整三情景渗透率 ----
    sc_cols = ["基准新能源占比"]
    for c in sc_cols:
        out[c] = out[c] * out["mult"]
        out[c] = out[c].clip(0, cap)

    # ---- 同步燃油占比 ----
    out["基准燃油占比"] = 1 - out["基准新能源占比"]

    # ---- 整理列顺序 ----
    out = (out[["city","distance_group","mult",
                "基准新能源占比","基准燃油占比"]]
           .sort_values(["city","distance_group"])
           .reset_index(drop=True))
    return out

# 用法示例：
new_energy_rate_2023 = build_ev_rate_by_distance(new_energy_rate_2023, normalize="by_10_50", cap=0.99)
new_energy_rate_2023

,city,distance_group,mult,基准新能源占比,基准燃油占比
0,七台河市,10-50,1.000000,0.004052,0.995948
1,七台河市,100-200,1.823270,0.007388,0.992612
2,七台河市,200-400,0.597938,0.002423,0.997577
3,七台河市,400+,0.821797,0.003330,0.996670
4,七台河市,50-100,0.375552,0.001522,0.998478
...,...,...,...,...,...
2191,龙岩市,100-200,1.823270,0.045322,0.954678
2192,龙岩市,200-400,0.597938,0.014863,0.985137
2193,龙岩市,400+,0.821797,0.020428,0.979572
2194,龙岩市,50-100,0.375552,0.009335,0.990665


## 1.3 不同路径对应里程区间新能源占比（rate_in_distances）:

In [59]:
rate_in_distances = pd.read_excel("../data/距离对应新能源占比.xlsx",sheet_name = "Sheet3")
# 加入距离分组列作为外键 连接该表到距离总流量表
bins = [10, 50, 100, 200, 400, np.inf]
labels = ['10-50','50-100', '100-200', '200-400', '400+']
rate_in_distances['distance_group'] = pd.cut(rate_in_distances['最小里程'], bins=bins, labels=labels, right=False)
rate_in_distances

,最小里程,最大里程,新能源占比,燃油车占比,distance_group
0,10,50.000000,0.170732,0.006432,10-50
1,50,100.000000,0.207371,0.144903,50-100
2,100,200.000000,0.408999,0.344497,100-200
3,200,400.000000,0.223975,0.374235,200-400
4,400,NaN,0.074886,0.211131,400+


In [61]:
ref_10_50 = (
    rate_in_distances
    .loc[rate_in_distances['distance_group'] == '10-50',
         ['新能源占比', '燃油车占比']]
    .mean()
)
new_row = pd.DataFrame({
    'distance_group': ['intra-city'],
    '新能源占比': [ref_10_50['新能源占比']],
    '燃油车占比': [ref_10_50['燃油车占比']]
})

rate_in_distances = pd.concat(
    [rate_in_distances, new_row],
    ignore_index=True
)
rate_in_distances


,最小里程,最大里程,新能源占比,燃油车占比,distance_group
0,10.000000,50.000000,0.170732,0.006432,10-50
1,50.000000,100.000000,0.207371,0.144903,50-100
2,100.000000,200.000000,0.408999,0.344497,100-200
3,200.000000,400.000000,0.223975,0.374235,200-400
4,400.000000,NaN,0.074886,0.211131,400+
5,NaN,NaN,0.170732,0.006432,intra-city


# 2.具体数据示例计算距离区间对应最终新能源率

**各DF对应的特征**

1.1 未来各个城市5距离区间的货运量（cities_flows_25）： 城市 城市总货运量 货运距离区间 城市不同距离区间货运量占比  城市不同距离区间货运量

1.2 不同情境下城市新能源渗透率（new_energy_rate_in_base/low/high）：不同年份情景下城市的新能源渗透率率（4年*3情景）

1.3 不同路径对应里程区间新能源占比（rate_in_distances）：不同路径对应里程区间如[0-50]新能源占比 

**选择上海市作为起始城市：获取上海市起始城市的2023新能源化率:基准情景**

## 2.1 Demo示例: 选择上海市计算不同距离等级新能源占比


**获取上海市不同距离总货运量**

In [62]:
sh_demo_cities_flows_23 = cities_flows_23.loc[(cities_flows_23.city == "上海市")]
sh_demo_cities_flows_23

,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en
402,上海市,2023,50436.000015,intra-city,0.572076,0.443156,22351.001765,Shanghai
403,上海市,2023,50436.000015,10-50,0.000000,0.000000,0.000000,Shanghai
404,上海市,2023,50436.000015,50-100,0.144142,0.304661,15365.889943,Shanghai
405,上海市,2023,50436.000015,100-200,0.096900,0.095260,4804.511323,Shanghai
406,上海市,2023,50436.000015,200-400,0.068252,0.066252,3341.485792,Shanghai
407,上海市,2023,50436.000015,400+,0.118630,0.090672,4573.111192,Shanghai


**获取上海市新能源占比**

In [63]:
sh_demo_new_energy_rate_2023 = new_energy_rate_2023.loc[(new_energy_rate_2023.city == "上海市")]
sh_demo_new_energy_rate_2023

,city,distance_group,mult,基准新能源占比,基准燃油占比
30,上海市,10-50,1.000000,0.217460,0.782540
31,上海市,100-200,1.823270,0.396488,0.603512
32,上海市,200-400,0.597938,0.130028,0.869972
33,上海市,400+,0.821797,0.178708,0.821292
34,上海市,50-100,0.375552,0.081668,0.918332
35,上海市,intra-city,1.000000,0.217460,0.782540


**获取距离对应新能源占比**

In [64]:
rate_in_distances

,最小里程,最大里程,新能源占比,燃油车占比,distance_group
0,10.000000,50.000000,0.170732,0.006432,10-50
1,50.000000,100.000000,0.207371,0.144903,50-100
2,100.000000,200.000000,0.408999,0.344497,100-200
3,200.000000,400.000000,0.223975,0.374235,200-400
4,400.000000,NaN,0.074886,0.211131,400+
5,NaN,NaN,0.170732,0.006432,intra-city


In [65]:
#合并不同距离新能源和燃油车占比 到 上海市不同距离总货运量
sh_demo_cities_flows_rate_in_distances_23 = pd.merge(sh_demo_cities_flows_23, rate_in_distances[['distance_group', '新能源占比', '燃油车占比']], on='distance_group', how='left')
sh_demo_cities_flows_rate_in_distances_23

,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en,新能源占比,燃油车占比
0,上海市,2023,50436.000015,intra-city,0.572076,0.443156,22351.001765,Shanghai,0.170732,0.006432
1,上海市,2023,50436.000015,10-50,0.000000,0.000000,0.000000,Shanghai,0.170732,0.006432
2,上海市,2023,50436.000015,50-100,0.144142,0.304661,15365.889943,Shanghai,0.207371,0.144903
3,上海市,2023,50436.000015,100-200,0.096900,0.095260,4804.511323,Shanghai,0.408999,0.344497
4,上海市,2023,50436.000015,200-400,0.068252,0.066252,3341.485792,Shanghai,0.223975,0.374235
5,上海市,2023,50436.000015,400+,0.118630,0.090672,4573.111192,Shanghai,0.074886,0.211131


In [66]:
#合并不同距离新能源和燃油车占比 上海市不同距离总货运量 和 起始城市新能源渗透率 以 计算 不同距离对应最终新能源率
sh_demo_df_23 = pd.merge(sh_demo_cities_flows_rate_in_distances_23, sh_demo_new_energy_rate_2023, on= ["city", "distance_group"], how='left')
sh_demo_df_23

,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en,新能源占比,燃油车占比,mult,基准新能源占比,基准燃油占比
0,上海市,2023,50436.000015,intra-city,0.572076,0.443156,22351.001765,Shanghai,0.170732,0.006432,1.000000,0.217460,0.782540
1,上海市,2023,50436.000015,10-50,0.000000,0.000000,0.000000,Shanghai,0.170732,0.006432,1.000000,0.217460,0.782540
2,上海市,2023,50436.000015,50-100,0.144142,0.304661,15365.889943,Shanghai,0.207371,0.144903,0.375552,0.081668,0.918332
3,上海市,2023,50436.000015,100-200,0.096900,0.095260,4804.511323,Shanghai,0.408999,0.344497,1.823270,0.396488,0.603512
4,上海市,2023,50436.000015,200-400,0.068252,0.066252,3341.485792,Shanghai,0.223975,0.374235,0.597938,0.130028,0.869972
5,上海市,2023,50436.000015,400+,0.118630,0.090672,4573.111192,Shanghai,0.074886,0.211131,0.821797,0.178708,0.821292


In [67]:
#计算最终值函数
def calc_final(row, col):
    r = row[col] #起始城市新能源渗透率
    a = row["新能源占比"] * r # 距离对应新能源占比 * 起始城市新能源渗透率
    b = row["燃油车占比"] * (1-r) # 距离对应新能源占比 * 起始城市燃油车渗透率
    return a / (a+b) if (a+b)!=0 else 0 # 最终总分配占比
sh_demo_df_23["最终基准新能源占比"] = sh_demo_df_23.apply(lambda x: calc_final(x,"基准新能源占比"), axis=1)

sh_demo_df_23

,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en,新能源占比,燃油车占比,mult,基准新能源占比,基准燃油占比,最终基准新能源占比
0,上海市,2023,50436.000015,intra-city,0.572076,0.443156,22351.001765,Shanghai,0.170732,0.006432,1.000000,0.217460,0.782540,0.880611
1,上海市,2023,50436.000015,10-50,0.000000,0.000000,0.000000,Shanghai,0.170732,0.006432,1.000000,0.217460,0.782540,0.880611
2,上海市,2023,50436.000015,50-100,0.144142,0.304661,15365.889943,Shanghai,0.207371,0.144903,0.375552,0.081668,0.918332,0.112900
3,上海市,2023,50436.000015,100-200,0.096900,0.095260,4804.511323,Shanghai,0.408999,0.344497,1.823270,0.396488,0.603512,0.438195
4,上海市,2023,50436.000015,200-400,0.068252,0.066252,3341.485792,Shanghai,0.223975,0.374235,0.597938,0.130028,0.869972,0.082106
5,上海市,2023,50436.000015,400+,0.118630,0.090672,4573.111192,Shanghai,0.074886,0.211131,0.821797,0.178708,0.821292,0.071648


## 2.2 整体计算距离区间对应最终新能源率

**获取各市不同距离总货运量**

In [68]:
cities_flows_23

,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en
0,北京市,2023,19399.000009,intra-city,0.407323,0.269966,5237.075531,Beijing
1,北京市,2023,19399.000009,10-50,0.034243,0.278118,5395.217434,Beijing
2,北京市,2023,19399.000009,50-100,0.000000,0.000000,0.000000,Beijing
3,北京市,2023,19399.000009,100-200,0.412020,0.346555,6722.818816,Beijing
4,北京市,2023,19399.000009,200-400,0.054448,0.045220,877.219386,Beijing
...,...,...,...,...,...,...,...,...
1705,哈密市,2023,34773.493677,10-50,0.000000,0.000000,0.000000,Hami
1706,哈密市,2023,34773.493677,50-100,0.000000,0.000000,0.000000,Hami
1707,哈密市,2023,34773.493677,100-200,0.000000,0.000000,0.000000,Hami
1708,哈密市,2023,34773.493677,200-400,0.000000,0.000000,0.000000,Hami


**获取各市对应新能源渗透率**

In [69]:
new_energy_rate_2023

,city,distance_group,mult,基准新能源占比,基准燃油占比
0,七台河市,10-50,1.000000,0.004052,0.995948
1,七台河市,100-200,1.823270,0.007388,0.992612
2,七台河市,200-400,0.597938,0.002423,0.997577
3,七台河市,400+,0.821797,0.003330,0.996670
4,七台河市,50-100,0.375552,0.001522,0.998478
...,...,...,...,...,...
2191,龙岩市,100-200,1.823270,0.045322,0.954678
2192,龙岩市,200-400,0.597938,0.014863,0.985137
2193,龙岩市,400+,0.821797,0.020428,0.979572
2194,龙岩市,50-100,0.375552,0.009335,0.990665


**获取距离对应新能源占比**

In [70]:
rate_in_distances

,最小里程,最大里程,新能源占比,燃油车占比,distance_group
0,10.000000,50.000000,0.170732,0.006432,10-50
1,50.000000,100.000000,0.207371,0.144903,50-100
2,100.000000,200.000000,0.408999,0.344497,100-200
3,200.000000,400.000000,0.223975,0.374235,200-400
4,400.000000,NaN,0.074886,0.211131,400+
5,NaN,NaN,0.170732,0.006432,intra-city


**合并不同距离新能源和燃油车占比 到 城市不同距离总货运量**

In [71]:
merged_df_23 = pd.merge(cities_flows_23, rate_in_distances[['distance_group', '新能源占比', '燃油车占比']], on='distance_group', how='left')
merged_df_23

,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en,新能源占比,燃油车占比
0,北京市,2023,19399.000009,intra-city,0.407323,0.269966,5237.075531,Beijing,0.170732,0.006432
1,北京市,2023,19399.000009,10-50,0.034243,0.278118,5395.217434,Beijing,0.170732,0.006432
2,北京市,2023,19399.000009,50-100,0.000000,0.000000,0.000000,Beijing,0.207371,0.144903
3,北京市,2023,19399.000009,100-200,0.412020,0.346555,6722.818816,Beijing,0.408999,0.344497
4,北京市,2023,19399.000009,200-400,0.054448,0.045220,877.219386,Beijing,0.223975,0.374235
...,...,...,...,...,...,...,...,...,...,...
1705,哈密市,2023,34773.493677,10-50,0.000000,0.000000,0.000000,Hami,0.170732,0.006432
1706,哈密市,2023,34773.493677,50-100,0.000000,0.000000,0.000000,Hami,0.207371,0.144903
1707,哈密市,2023,34773.493677,100-200,0.000000,0.000000,0.000000,Hami,0.408999,0.344497
1708,哈密市,2023,34773.493677,200-400,0.000000,0.000000,0.000000,Hami,0.223975,0.374235


**合并不同距离新能源和燃油车占比 各市不同距离总货运量 和 起始城市新能源渗透率 以 计算 不同距离对应最终新能源率**

In [72]:
#在以上基础上再加入起始城市新能源渗透率
df_23 = pd.merge(merged_df_23, new_energy_rate_2023, on=["city", "distance_group"], how='left')
df_23

,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en,新能源占比,燃油车占比,mult,基准新能源占比,基准燃油占比
0,北京市,2023,19399.000009,intra-city,0.407323,0.269966,5237.075531,Beijing,0.170732,0.006432,1.000000,0.165930,0.834070
1,北京市,2023,19399.000009,10-50,0.034243,0.278118,5395.217434,Beijing,0.170732,0.006432,1.000000,0.165930,0.834070
2,北京市,2023,19399.000009,50-100,0.000000,0.000000,0.000000,Beijing,0.207371,0.144903,0.375552,0.062315,0.937685
3,北京市,2023,19399.000009,100-200,0.412020,0.346555,6722.818816,Beijing,0.408999,0.344497,1.823270,0.302535,0.697465
4,北京市,2023,19399.000009,200-400,0.054448,0.045220,877.219386,Beijing,0.223975,0.374235,0.597938,0.099216,0.900784
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1705,哈密市,2023,34773.493677,10-50,0.000000,0.000000,0.000000,Hami,0.170732,0.006432,1.000000,0.010093,0.989907
1706,哈密市,2023,34773.493677,50-100,0.000000,0.000000,0.000000,Hami,0.207371,0.144903,0.375552,0.003790,0.996210
1707,哈密市,2023,34773.493677,100-200,0.000000,0.000000,0.000000,Hami,0.408999,0.344497,1.823270,0.018402,0.981598
1708,哈密市,2023,34773.493677,200-400,0.000000,0.000000,0.000000,Hami,0.223975,0.374235,0.597938,0.006035,0.993965


In [73]:
#计算最终占比函数
def calc_final(row, col):
    r = row[col] #起始城市新能源渗透率
    a = row["新能源占比"] * r # 距离对应新能源占比 * 起始城市新能源渗透率
    b = row["燃油车占比"] * (1-r) # 距离对应新能源占比 * 起始城市燃油车渗透率
    return a / (a+b) if (a+b)!=0 else 0 # 最终总分配占比
#计算25年最终占比
df_23["最终基准新能源占比"] = df_23.apply(lambda x: calc_final(x,"基准新能源占比"), axis=1)
df_23

,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en,新能源占比,燃油车占比,mult,基准新能源占比,基准燃油占比,最终基准新能源占比
0,北京市,2023,19399.000009,intra-city,0.407323,0.269966,5237.075531,Beijing,0.170732,0.006432,1.000000,0.165930,0.834070,0.840775
1,北京市,2023,19399.000009,10-50,0.034243,0.278118,5395.217434,Beijing,0.170732,0.006432,1.000000,0.165930,0.834070,0.840775
2,北京市,2023,19399.000009,50-100,0.000000,0.000000,0.000000,Beijing,0.207371,0.144903,0.375552,0.062315,0.937685,0.086847
3,北京市,2023,19399.000009,100-200,0.412020,0.346555,6722.818816,Beijing,0.408999,0.344497,1.823270,0.302535,0.697465,0.339925
4,北京市,2023,19399.000009,200-400,0.054448,0.045220,877.219386,Beijing,0.223975,0.374235,0.597938,0.099216,0.900784,0.061843
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1705,哈密市,2023,34773.493677,10-50,0.000000,0.000000,0.000000,Hami,0.170732,0.006432,1.000000,0.010093,0.989907,0.212984
1706,哈密市,2023,34773.493677,50-100,0.000000,0.000000,0.000000,Hami,0.207371,0.144903,0.375552,0.003790,0.996210,0.005416
1707,哈密市,2023,34773.493677,100-200,0.000000,0.000000,0.000000,Hami,0.408999,0.344497,1.823270,0.018402,0.981598,0.021772
1708,哈密市,2023,34773.493677,200-400,0.000000,0.000000,0.000000,Hami,0.223975,0.374235,0.597938,0.006035,0.993965,0.003621


## 2.3 整体计算距离区间对应不同情景下最终新能源和燃油车货运量

In [74]:
#按照最终比例计算最终新能源和燃油车在每个城市对应距离的货运量
# columns[4] -> weight_distance_2025/30
def fianl_weight(df)-> pd.DataFrame:
    df["基准新能源货运量"] = df[df.columns[6]] * df["最终基准新能源占比"]
    df["基准燃油车货运量"] = df[df.columns[6]]  - df["基准新能源货运量"]
    return df

In [75]:
fianl_weight_23 = fianl_weight(df_23)
fianl_weight_23

,city,year,total_ton,distance_group,ton_share_orig,ton_share_adj,weight_distance,city_en,新能源占比,燃油车占比,mult,基准新能源占比,基准燃油占比,最终基准新能源占比,基准新能源货运量,基准燃油车货运量
0,北京市,2023,19399.000009,intra-city,0.407323,0.269966,5237.075531,Beijing,0.170732,0.006432,1.000000,0.165930,0.834070,0.840775,4403.204351,833.871180
1,北京市,2023,19399.000009,10-50,0.034243,0.278118,5395.217434,Beijing,0.170732,0.006432,1.000000,0.165930,0.834070,0.840775,4536.166175,859.051259
2,北京市,2023,19399.000009,50-100,0.000000,0.000000,0.000000,Beijing,0.207371,0.144903,0.375552,0.062315,0.937685,0.086847,0.000000,0.000000
3,北京市,2023,19399.000009,100-200,0.412020,0.346555,6722.818816,Beijing,0.408999,0.344497,1.823270,0.302535,0.697465,0.339925,2285.255773,4437.563043
4,北京市,2023,19399.000009,200-400,0.054448,0.045220,877.219386,Beijing,0.223975,0.374235,0.597938,0.099216,0.900784,0.061843,54.249812,822.969574
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1705,哈密市,2023,34773.493677,10-50,0.000000,0.000000,0.000000,Hami,0.170732,0.006432,1.000000,0.010093,0.989907,0.212984,0.000000,0.000000
1706,哈密市,2023,34773.493677,50-100,0.000000,0.000000,0.000000,Hami,0.207371,0.144903,0.375552,0.003790,0.996210,0.005416,0.000000,0.000000
1707,哈密市,2023,34773.493677,100-200,0.000000,0.000000,0.000000,Hami,0.408999,0.344497,1.823270,0.018402,0.981598,0.021772,0.000000,0.000000
1708,哈密市,2023,34773.493677,200-400,0.000000,0.000000,0.000000,Hami,0.223975,0.374235,0.597938,0.006035,0.993965,0.003621,0.000000,0.000000


In [86]:
fianl_weight_23.to_excel(
    "../out/dataset3_PowertrainShare_2023年距离对应新能源和燃油车货运占比.xlsx",
    index=False,
    engine="openpyxl"
)
